# SASRec BPI2012 Colab Sanity Check (Eval Fix)

Colab notebook for sanity checking the two selected post-fix baseline candidates.

Goals:
- reuse already completed runs instead of retraining them
- train only the missing seeds for the two selected candidate settings
- summarize mean/std and valid-test trends separately for `NDCG@10` and `NDCG@5` model-selection criteria


In [ ]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('NDCG10_OUTPUT_DIR:', NDCG10_OUTPUT_DIR)
print('NDCG5_OUTPUT_DIR:', NDCG5_OUTPUT_DIR)


In [ ]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$NDCG10_OUTPUT_DIR"
!mkdir -p "$NDCG5_OUTPUT_DIR"


In [ ]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


In [ ]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


In [ ]:
!pip install -r requirements_colab.txt


In [ ]:
!ls "$DATA_DIR"


In [ ]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only


## Candidate settings

Selected candidates for sanity check:
- `anchor_ml20` (`hidden_units=32, maxlen=20, dropout=0.2`)
- `refine_ml50_do035` (`hidden_units=50, maxlen=50, dropout=0.35`)

We will evaluate them under two model-selection criteria separately:
- `full_valid_ndcg@10`
- `full_valid_ndcg@5`


## Check existing completed runs

These runs should already exist and **must not be retrained**.


In [ ]:
from pathlib import Path

existing_ndcg10 = [
    'anchor_ml20_s42',
    'refine_ml50_do035_s42',
]
existing_ndcg5 = [
    'anchor_ml20_s42',
    'refine_ml50_do035_s42',
]

for label, output_dir, run_names in [
    ('NDCG@10', Path(NDCG10_OUTPUT_DIR), existing_ndcg10),
    ('NDCG@5', Path(NDCG5_OUTPUT_DIR), existing_ndcg5),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


## Train only missing seeds for `NDCG@10`

Run these cells only if the corresponding run directory does not already exist.


### anchor_ml20_s2024


In [ ]:
!python src/train_sasrec.py \
  --run_name anchor_ml20_s2024 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### anchor_ml20_s7


In [ ]:
!python src/train_sasrec.py \
  --run_name anchor_ml20_s7 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### refine_ml50_do035_s2024


In [ ]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do035_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### refine_ml50_do035_s7


In [ ]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do035_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


## Train only missing seeds for `NDCG@5`

Run these cells only if the corresponding run directory does not already exist.


### anchor_ml20_s2024


In [ ]:
!python src/train_sasrec.py \
  --run_name anchor_ml20_s2024 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### anchor_ml20_s7


In [ ]:
!python src/train_sasrec.py \
  --run_name anchor_ml20_s7 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### refine_ml50_do035_s2024


In [ ]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do035_s2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 2024 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### refine_ml50_do035_s7


In [ ]:
!python src/train_sasrec.py \
  --run_name refine_ml50_do035_s7 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --seed 7 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


## Rebuild result table from run folders

This avoids schema issues in `experiment_index.csv` and lets us combine existing and newly added runs safely.


In [ ]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
        }
        best_valid = summary.get('best_valid', {})
        best_test = summary.get('best_test_at_best_valid', {})
        def pick(metrics_group, mode, key):
            return metrics_group.get(mode, {}).get(key)
        row.update({
            'best_valid_full_ndcg@10': pick(best_valid, 'full', 'ndcg@10'),
            'best_valid_full_hr@10': pick(best_valid, 'full', 'hr@10'),
            'best_valid_full_ndcg@5': pick(best_valid, 'full', 'ndcg@5'),
            'best_valid_full_hr@5': pick(best_valid, 'full', 'hr@5'),
            'best_valid_full_mrr': pick(best_valid, 'full', 'mrr'),
            'best_test_full_ndcg@10': pick(best_test, 'full', 'ndcg@10'),
            'best_test_full_hr@10': pick(best_test, 'full', 'hr@10'),
            'best_test_full_ndcg@5': pick(best_test, 'full', 'ndcg@5'),
            'best_test_full_hr@5': pick(best_test, 'full', 'hr@5'),
            'best_test_full_mrr': pick(best_test, 'full', 'mrr'),
            'best_valid_sampled_ndcg@10': pick(best_valid, 'sampled', 'ndcg@10'),
            'best_valid_sampled_hr@10': pick(best_valid, 'sampled', 'hr@10'),
            'best_valid_sampled_ndcg@5': pick(best_valid, 'sampled', 'ndcg@5'),
            'best_valid_sampled_hr@5': pick(best_valid, 'sampled', 'hr@5'),
            'best_valid_sampled_mrr': pick(best_valid, 'sampled', 'mrr'),
            'best_test_sampled_ndcg@10': pick(best_test, 'sampled', 'ndcg@10'),
            'best_test_sampled_hr@10': pick(best_test, 'sampled', 'hr@10'),
            'best_test_sampled_ndcg@5': pick(best_test, 'sampled', 'ndcg@5'),
            'best_test_sampled_hr@5': pick(best_test, 'sampled', 'hr@5'),
            'best_test_sampled_mrr': pick(best_test, 'sampled', 'mrr'),
        })
        rows.append(row)
    return pd.DataFrame(rows)


## `NDCG@10` sanity check summary


In [ ]:
ndcg10_targets = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]

df10 = rebuild_df(NDCG10_OUTPUT_DIR)
df10_sc = df10[df10['run_name'].isin(ndcg10_targets)].copy()
df10_sc = df10_sc.sort_values(['run_name']).reset_index(drop=True)
df10_sc[[
    'run_name', 'seed',
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]]


In [ ]:
df10_sc['candidate'] = df10_sc['run_name'].apply(
    lambda x: 'anchor_ml20' if 'anchor_ml20' in x else 'refine_ml50_do035'
)
summary10 = df10_sc.groupby('candidate')[[
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary10


Interpretation guide for `NDCG@10`:
- compare mean/std of `best_valid_full_ndcg@10` and `best_test_full_ndcg@10`
- then check whether `@5`, sampled, and MRR show a similar trend


## `NDCG@5` sanity check summary


In [ ]:
ndcg5_targets = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
    'refine_ml50_do035_s42',
    'refine_ml50_do035_s2024',
    'refine_ml50_do035_s7',
]

df5 = rebuild_df(NDCG5_OUTPUT_DIR)
df5_sc = df5[df5['run_name'].isin(ndcg5_targets)].copy()
df5_sc = df5_sc.sort_values(['run_name']).reset_index(drop=True)
df5_sc[[
    'run_name', 'seed',
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]]


In [ ]:
df5_sc['candidate'] = df5_sc['run_name'].apply(
    lambda x: 'anchor_ml20' if 'anchor_ml20' in x else 'refine_ml50_do035'
)
summary5 = df5_sc.groupby('candidate')[[
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary5


Interpretation guide for `NDCG@5`:
- compare mean/std of `best_valid_full_ndcg@5` and `best_test_full_ndcg@5`
- then check whether `@10`, sampled, and MRR show a similar trend
